# Configuration


In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(".env", override=True)  # Change to ".env.simulation" for simulation data

# Load configuration from .env
simulation_root = os.getenv("SIMULATION_DIR")
converted_dir = os.getenv("CONVERT_TARGET_DIR")
batch_checkpoint_dir = os.getenv("BATCH_CHECKPOINT_DIR")
summarized_dir = os.getenv("SUMMARIZED_TARGET_DIR")
arena_heatmaps_output = os.getenv("ARENA_HEATMAP_TARGET_DIR")

# Display loaded configuration
print("Configuration loaded from .env:")
print(f"  SIMULATION_DIR: {simulation_root}")
print(f"  CONVERT_TARGET_DIR: {converted_dir}")
print(f"  BATCH_CHECKPOINT_DIR: {batch_checkpoint_dir}")
print(f"  SUMMARIZED_TARGET_DIR: {summarized_dir}")
print(f"  ARENA_HEATMAP_TARGET_DIR: {arena_heatmaps_output}")

# Create directories if they don't exist
os.makedirs(converted_dir, exist_ok=True)
os.makedirs(batch_checkpoint_dir, exist_ok=True)
os.makedirs(summarized_dir, exist_ok=True)
os.makedirs(arena_heatmaps_output, exist_ok=True)

# Data Compiling

## Convert Simulation Log to Parquet / CSV

In [ ]:
from compile.log_to_parquet import ( 
    convert_all_configs
)

convert_all_configs(simulation_root, converted_dir)

## Generate Summarization CSV

### Generate Batched CSV

Process CSVs in batches and save checkpoints

Structure: base_dir/BotA_vs_BotB/ConfigFolder/*.csv

In [ ]:
from compile.generator import batch_process_pacing_segments

import time

timebin_size = 1
batch_size = 4 # if there's 156 matchup simulation folder, it will generate 156 / 2 = 78 summarization batch csv
input_format = "auto"  # "csv", "parquet", or "auto" (auto prefers parquet over csv)

start = time.time()

batch_process_pacing_segments(
    converted_dir, 
    batch_size=batch_size,
    applied_bots=["GA", "MCTS","NN"],
    min_pacing=0.1,
    max_pacing=0.6,
    checkpoint_dir=batch_checkpoint_dir)

elapsed_seconds = time.time() - start
hours, remainder = divmod(elapsed_seconds, 3600)
minutes, seconds = divmod(remainder, 60)
processing_time = f"{int(hours):02d}:{int(minutes):02d}:{seconds:.2f}"
print(f"\nProcessing Time: {processing_time}")

### Generate Final Summarization CSV from Batches

Generate timebin summaries from batched timebin checkpoints
Loads batch files and creates final summaries

In [ ]:
from compile.generator import generate_pacing_segments_from_batches

generate_pacing_segments_from_batches(batch_checkpoint_dir, summarized_dir)

# Plotting

In [ ]:
from plotting.pacing_target_analyzer import plot_all_pacing_targets
import pandas as pd

df_pacing_segments = pd.read_parquet(f"{summarized_dir}/summary_pacing_segments.parquet")

figs = plot_all_pacing_targets(df_pacing_segments, output_dir=f"{summarized_dir}/pacing_target_charts")